In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
import json

In [ ]:
label = 3

In [ ]:
with open('../data/ZigZag_10_90_deg/ZigZag_10_90_deg{}.json'.format(label), 'r') as f:
    data = json.load(f)
    fusedVertices = data['FusedVertices']
    fusedVertices = [True if vx == 1 else False for vx in fusedVertices]
    V = data['Vertices']
    F = data['Faces']
    m = MeshFEM.Mesh(V, F)
    ipu = inflation.InflatablePeriodicUnit(m, fusedVertices)

    finalMarkers = np.where(np.array(fusedVertices) == 1)[0]
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

    fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
    ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)
    pointList = np.where(np.array(fusedVtx) == 1)
    visualization.plot_2d_mesh(m, pointList=pointList, width=5, height=5)
    plt.savefig("zigzag_pattern.svg", dpi = 300)

In [ ]:

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
dofs = np.load("../experiments/output/zigzag_line_10_to_90/2023_12_04_20_23//3/zigzag_line_10_to_90_dofs_before_stiffness_3.npy")

In [ ]:
len(dofs)

In [ ]:
# ipu.setVars(dofs)

In [ ]:
ipu.numVars()

In [ ]:
viewer.update()

In [ ]:
allow_bending = False
useTFT = True
disableFusedRegionTFT = False
stiffness_pressure = 0.1

In [ ]:

hessianShiftForRigidMotion = 1e-10
hessianShiftForAlphainPlanar = 1e-12
# Choose strategy for constraining rigid motion
# We first use no fix vars to check whether the equilibrium converge to a planar state, use a hessian shift rather than applying fixed vars to constrain the rigid motion so that the equilibrium solve converges faster. 
bending_fixed_vars = [] if allow_bending else [ipu.numVars() - 2]
fixedVars, hessianShift = bending_fixed_vars, hessianShiftForRigidMotion

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

opts.niter = 100
opts.gradTol = 1e-10

def cb(it):
    # if it % framerate == 0:
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
        # viewer.update()

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

opts.niter = 1000
# Solve for true equilibrium with a much smaller hessian shift that's only used for the alpha variable when kappa becomes zero. Pin down the the x, y, z value of a center vertex. 
fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + bending_fixed_vars, hessianShiftForAlphainPlanar
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
